# Bayesian Networks & Autoregressive Language Models

**AI Laboratory** — connecting Bayesian networks, the chain rule, and autoregressive language
modelling.

The chain-rule factorisation underlies autoregressive text generation:

$$P(x_1,\dots,x_T) = P(x_1)\prod_{t=2}^{T} P(x_t \mid x_1,\dots,x_{t-1}).$$

A first-order Markov language model approximates each factor by $P(x_t\mid x_{t-1})$ — a very
simple Bayesian network $X_1 \to X_2 \to \cdots \to X_T$. This notebook builds such models from a
tiny dataset, tests their probabilistic invariants, generates text, extends to a second-order
model, and answers Questions 1–14.

> **Workflow:** Understand → Design → Ask the LLM → Implement → Test → Reflect. The LLM is a tool
> for constructing the system, not a replacement for understanding it.


## Part I — From Probability to Language

For *the cat sat on the mat*, the chain rule gives
$P(X_1,\dots,X_6)=P(X_1)P(X_2\mid X_1)\cdots P(X_6\mid X_1,\dots,X_5)$.

**Question 1 — Why is this decomposition useful for generating text?**
It breaks the intractable joint distribution over whole sentences into a sequence of simple
*next-token* distributions. We can generate left-to-right: sample $x_1$, then sample each $x_t$
from $P(x_t\mid x_{<t})$, appending as we go. Each step only requires a conditional over the
vocabulary, which is something we can estimate and sample from — so a hard joint-modelling problem
becomes a sequence of easy conditional-modelling problems.


## Part II — A Bayesian Network for Text

If each word depends only on the immediately preceding word, the network is the chain
$X_1\to X_2\to X_3\to X_4$, representing
$P(X_1,X_2,X_3,X_4)=P(X_1)P(X_2\mid X_1)P(X_3\mid X_2)P(X_4\mid X_3)$ — a **first-order Markov**
model.

**Question 2 — What independence assumption is made?**
Each token is conditionally independent of all tokens before the previous one, given the previous
token:
$$P(X_t \mid X_1,\dots,X_{t-1}) = P(X_t \mid X_{t-1}).$$
Equivalently, $X_t \perp \{X_1,\dots,X_{t-2}\} \mid X_{t-1}$.


## Part III — Build a Small Language Dataset

Lower-cased, each word a token, with `<START>` / `<END>` markers per sentence.


In [1]:
RAW = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

START, END = "<START>", "<END>"

def tokenise(sentences):
    """Lower-case and wrap each sentence with START/END markers."""
    return [[START] + s.lower().split() + [END] for s in sentences]

corpus = tokenise(RAW)
for row in corpus:
    print(" ".join(row))


<START> the cat sat on the mat <END>
<START> the cat sat on the rug <END>
<START> the dog sat on the mat <END>
<START> the dog ran to the park <END>
<START> the cat ran to the park <END>
<START> the dog sat on the rug <END>


## Part IV — Constructing the Conditional Probability Table (first-order)

The model estimates $P(X_t\mid X_{t-1})=\dfrac{C(w_i,w_j)}{\sum_k C(w_i,w_k)}$, where $C(w_i,w_j)$
counts how often $w_j$ follows $w_i$.

> **Note on the PDF's example.** The PDF illustrates with "$P(\text{cat}\mid\text{the})=3/5$,
> $P(\text{dog}\mid\text{the})=2/5$". That is a *hypothetical* sub-example, not the figure for this
> dataset: here `the` is also followed by `mat`, `rug`, and `park`, and it precedes `cat` and
> `dog` three times each. We compute the **actual** distribution below.


In [2]:
from collections import defaultdict, Counter

def count_transitions(corpus):
    """C(w_i, w_j): counts[prev][next] = number of times next follows prev."""
    counts = defaultdict(Counter)
    for sent in corpus:
        for prev, nxt in zip(sent, sent[1:]):
            counts[prev][nxt] += 1
    return counts

def build_cpt(counts):
    """Normalise counts into P(next | prev)."""
    cpt = {}
    for prev, nexts in counts.items():
        total = sum(nexts.values())
        cpt[prev] = {w: c / total for w, c in nexts.items()}
    return cpt

counts1 = count_transitions(corpus)
cpt1 = build_cpt(counts1)

def show_cpd(cpt, word):
    dist = cpt.get(word, {})
    items = sorted(dist.items(), key=lambda kv: (-kv[1], kv[0]))
    print(f"P(next | {word!r}):")
    for w, p in items:
        print(f"    {w:<10} {p:.3f}")

for w in ["the", "cat", "dog", "sat", "ran"]:
    show_cpd(cpt1, w)
    print()


P(next | 'the'):
    cat        0.250
    dog        0.250
    mat        0.167
    park       0.167
    rug        0.167

P(next | 'cat'):
    sat        0.667
    ran        0.333

P(next | 'dog'):
    sat        0.667
    ran        0.333

P(next | 'sat'):
    on         1.000

P(next | 'ran'):
    to         1.000



### Question 3 — the distributions and zero-probability transitions

The table above is $P(\text{next}\mid\text{current})$ for `the, cat, dog, sat, ran`. Observations:

* `P(cat|the) = P(dog|the) = 3/12 = 0.25`; `the` is also followed by `mat, rug, park`.
* `cat` and `dog` are **always** followed by `sat` or `ran` in this corpus — e.g. `P(sat|cat)`
  and `P(ran|cat)` are non-zero while `P(mat|cat)=0`.
* **Zero-probability transitions** are every pair never observed: e.g. `P(cat|cat)=0`,
  `P(the|the)=0`, `P(dog|sat)=0`, `P(park|ran)` depends on the data, etc. Any (current, next)
  pair absent from the counts has probability 0 under this maximum-likelihood estimate — a known
  limitation (unseen bigrams can never be generated).


## Part V / VI — Implement and Inspect the Model

The class below is exactly the behavioural specification from the PDF: count transitions, build
$P(X_t\mid X_{t-1})$, show a distribution, predict the most probable next token, and generate by
sampling until `<END>`. We then answer the inspection questions.


In [3]:
import random

class FirstOrderLM:
    """First-order autoregressive (bigram) language model from counts."""
    def __init__(self):
        self.counts = defaultdict(Counter)   # Q4: transition counts stored here
        self.cpt = {}

    def train(self, corpus):
        self.counts = count_transitions(corpus)
        self.cpt = build_cpt(self.counts)    # Q5: P(Xt|Xt-1) computed here (normalisation)
        return self

    def distribution(self, prev):
        return self.cpt.get(prev, {})

    def predict(self, prev):
        """Most probable next token: argmax_w P(w | prev)."""
        dist = self.distribution(prev)
        if not dist:
            return None
        return max(dist.items(), key=lambda kv: (kv[1], kv[0]))[0]

    def sample_next(self, prev, rng):
        """Sample next token from P(w | prev)."""
        dist = self.distribution(prev)
        if not dist:
            return None
        words, probs = zip(*dist.items())
        return rng.choices(words, weights=probs, k=1)[0]

    def generate(self, rng, greedy=False, max_len=20):
        """Generate a sentence. greedy=True -> argmax; else sample."""
        out, prev = [], START
        for _ in range(max_len):
            nxt = self.predict(prev) if greedy else self.sample_next(prev, rng)
            if nxt is None or nxt == END:
                break
            out.append(nxt)
            prev = nxt
        return out

lm1 = FirstOrderLM().train(corpus)
print("Most probable token after <START>:", lm1.predict(START))
print("Most probable token after 'the'  :", lm1.predict("the"))
print("Most probable token after 'sat'  :", lm1.predict("sat"))


Most probable token after <START>: the
Most probable token after 'the'  : dog
Most probable token after 'sat'  : on


### Inspection answers (Questions 4–7)

* **Q4 — Where are transition counts stored?** In `self.counts`, a `defaultdict(Counter)` mapping
  `prev -> Counter(next -> count)` (built by `count_transitions`).
* **Q5 — Where is $P(X_t\mid X_{t-1})$ computed?** In `build_cpt` (called from `train`), which
  divides each count by the row total; the result lives in `self.cpt`.
* **Q6 — How is the next word chosen?** Two modes: `predict` **always takes the argmax** (most
  probable word, deterministic), while `sample_next` **samples from the distribution**
  (stochastic). The difference: argmax always returns the single most likely continuation and so
  produces identical, low-diversity text; sampling draws proportionally to probability, giving
  varied text that still respects the model.
* **Q7 — Unseen word?** If `prev` has no recorded transitions, `distribution` returns `{}` and
  both `predict` and `sample_next` return `None`; generation stops. A maximum-likelihood bigram
  model assigns zero probability to unseen contexts (motivating smoothing in real models).


## Part VII — Test the Probability Model

A correct model must satisfy $\sum_v P(v\mid w)=1$ for every current word $w$.


In [4]:
def check_normalisation(cpt):
    ok = True
    for word in sorted(cpt):
        total = sum(cpt[word].values())
        flag = "OK" if abs(total - 1.0) < 1e-9 else "** NOT 1 **"
        print(f"  sum P(.|{word:<8}) = {total:.6f}  {flag}")
        ok = ok and abs(total - 1.0) < 1e-9
    return ok

print("First-order normalisation check:")
all_ok = check_normalisation(cpt1)
print("\nAll rows sum to 1:", all_ok)


First-order normalisation check:
  sum P(.|<START> ) = 1.000000  OK
  sum P(.|cat     ) = 1.000000  OK
  sum P(.|dog     ) = 1.000000  OK
  sum P(.|mat     ) = 1.000000  OK
  sum P(.|on      ) = 1.000000  OK
  sum P(.|park    ) = 1.000000  OK
  sum P(.|ran     ) = 1.000000  OK
  sum P(.|rug     ) = 1.000000  OK
  sum P(.|sat     ) = 1.000000  OK
  sum P(.|the     ) = 1.000000  OK
  sum P(.|to      ) = 1.000000  OK

All rows sum to 1: True


**Question 8 — If one total were 0.87, what would that tell you?**
That the implementation is **buggy**: a conditional distribution must sum to 1. A total of 0.87
means probability mass is missing — e.g. dividing by the wrong denominator, dropping some
successor counts, double-counting the normaliser, or mishandling the `<END>` token. It is a
property-based test that catches implementation errors the generated text alone would hide.


## Part VIII — Predicting the Next Word

In [5]:
for context in ["the", "cat", "dog", "sat", "ran", START]:
    dist = lm1.distribution(context)
    if not dist:
        print(f"P(.|{context}): <no successors>")
        continue
    pretty = ", ".join(f"{w}:{p:.2f}" for w, p in
                       sorted(dist.items(), key=lambda kv: (-kv[1], kv[0])))
    print(f"P(.|{context:<8}) = {{{pretty}}}   argmax = {lm1.predict(context)}")


P(.|the     ) = {cat:0.25, dog:0.25, mat:0.17, park:0.17, rug:0.17}   argmax = dog
P(.|cat     ) = {sat:0.67, ran:0.33}   argmax = sat
P(.|dog     ) = {sat:0.67, ran:0.33}   argmax = sat
P(.|sat     ) = {on:1.00}   argmax = on
P(.|ran     ) = {to:1.00}   argmax = to
P(.|<START> ) = {the:1.00}   argmax = the


**Question 9 — Are the most-probable predictions always what you'd expect?**
Not necessarily. The model's argmax reflects **corpus frequencies**, not linguistic intuition.
With ties (e.g. `cat` and `dog` both at 0.25 after `the`) the "most probable" word is decided
arbitrarily by the tie-break, and rare-but-sensible continuations are invisible. This shows a
**probability model** captures statistical regularities of the *training data*, which may diverge
from a human's broader linguistic expectations — especially with a tiny corpus.


## Part IX — Generate Text (sampling)

In [6]:
rng = random.Random(42)   # fixed seed for reproducibility
print("20 sampled sentences:\n")
generated = []
for i in range(20):
    sent = " ".join(lm1.generate(rng, greedy=False))
    generated.append(sent)
    print(f"{i+1:2d}. {sent}")


20 sampled sentences:

 1. the cat sat on the dog ran to the cat sat on the cat sat on the dog ran to
 2. the dog sat on the mat
 3. the park
 4. the dog sat on the rug
 5. the park
 6. the cat sat on the cat sat on the mat
 7. the mat
 8. the dog sat on the mat
 9. the rug
10. the dog sat on the mat
11. the park
12. the mat
13. the mat
14. the mat
15. the mat
16. the rug
17. the cat sat on the cat sat on the park
18. the dog ran to the dog sat on the park
19. the rug
20. the park


## Part X — Deterministic (greedy) vs Probabilistic (sampling)

In [7]:
print("Mode A — greedy (argmax), 5 sentences:")
for i in range(5):
    print("  ", " ".join(lm1.generate(rng, greedy=True)))

print("\nMode B — sampling, 5 sentences:")
for i in range(5):
    print("  ", " ".join(lm1.generate(rng, greedy=False)))


Mode A — greedy (argmax), 5 sentences:
   the dog sat on the dog sat on the dog sat on the dog sat on the dog sat on
   the dog sat on the dog sat on the dog sat on the dog sat on the dog sat on
   the dog sat on the dog sat on the dog sat on the dog sat on the dog sat on
   the dog sat on the dog sat on the dog sat on the dog sat on the dog sat on
   the dog sat on the dog sat on the dog sat on the dog sat on the dog sat on

Mode B — sampling, 5 sentences:
   the dog sat on the dog sat on the cat ran to the mat
   the park
   the cat ran to the rug
   the park
   the rug


**Question 10 — Which mode produces more variation, and why?**
**Sampling** (Mode B). Greedy generation always picks the single argmax at each step, so from a
given start it follows one fixed, highest-probability path and repeats the *same* sentence every
time (zero variation, and it can get stuck in a short loop/cycle). Sampling draws each next token
in proportion to its probability, so different runs explore different branches — producing varied
sentences while still respecting the learned distribution. Variation is the direct consequence of
injecting randomness at each conditional draw.


## Part XI / XII — A Second-Order Bayesian Network

Now condition on the **two** previous tokens: $P(X_t\mid X_{t-2},X_{t-1})$, with structure
$X_{t-2}\to X_t \leftarrow X_{t-1}$ and
$P(X_1,X_2,X_3,X_4)=P(X_1)P(X_2\mid X_1)P(X_3\mid X_1,X_2)P(X_4\mid X_2,X_3)$.

Implemented from counts of observed **triples** (trigram counts).


In [8]:
class SecondOrderLM:
    """Second-order (trigram) model: P(Xt | Xt-2, Xt-1) from triple counts."""
    def __init__(self):
        self.counts = defaultdict(Counter)   # (prev2, prev1) -> Counter(next)
        self.cpt = {}

    def train(self, corpus):
        for sent in corpus:
            padded = [START] + sent          # extra START so the first real token has a 2-context
            for a, b, c in zip(padded, padded[1:], padded[2:]):
                self.counts[(a, b)][c] += 1
        self.cpt = {ctx: {w: n / sum(nx.values()) for w, n in nx.items()}
                    for ctx, nx in self.counts.items()}
        return self

    def distribution(self, ctx):
        return self.cpt.get(ctx, {})

    def generate(self, rng, greedy=False, max_len=20):
        out, prev2, prev1 = [], START, START
        for _ in range(max_len):
            dist = self.distribution((prev2, prev1))
            if not dist:
                break
            if greedy:
                nxt = max(dist.items(), key=lambda kv: (kv[1], kv[0]))[0]
            else:
                words, probs = zip(*dist.items())
                nxt = rng.choices(words, weights=probs, k=1)[0]
            if nxt == END:
                break
            out.append(nxt)
            prev2, prev1 = prev1, nxt
        return out

lm2 = SecondOrderLM().train(corpus)

print("Example second-order distributions:")
for ctx in [(START, "the"), ("the", "cat"), ("the", "dog"), ("sat", "on"), ("on", "the")]:
    dist = lm2.distribution(ctx)
    pretty = ", ".join(f"{w}:{p:.2f}" for w, p in
                       sorted(dist.items(), key=lambda kv: (-kv[1], kv[0])))
    print(f"  P(.|{ctx}) = {{{pretty}}}")


Example second-order distributions:
  P(.|('<START>', 'the')) = {cat:0.50, dog:0.50}
  P(.|('the', 'cat')) = {sat:0.67, ran:0.33}
  P(.|('the', 'dog')) = {sat:0.67, ran:0.33}
  P(.|('sat', 'on')) = {the:1.00}
  P(.|('on', 'the')) = {mat:0.50, rug:0.50}


In [9]:
print("Second-order normalisation check:")
ok2 = all(abs(sum(d.values()) - 1.0) < 1e-9 for d in lm2.cpt.values())
print("  all contexts sum to 1:", ok2)

print("\n10 sampled sentences (second-order):\n")
rng2 = random.Random(7)
for i in range(10):
    print(f"  {i+1:2d}.", " ".join(lm2.generate(rng2, greedy=False)))


Second-order normalisation check:
  all contexts sum to 1: True

10 sampled sentences (second-order):

   1. the cat sat on the mat
   2. the cat sat on the mat
   3. the cat sat on the mat
   4. the dog sat on the mat
   5. the dog sat on the mat
   6. the dog sat on the mat
   7. the dog sat on the rug
   8. the dog sat on the mat
   9. the dog ran to the park
  10. the dog sat on the mat


**Question 11 — First-order vs second-order differences**

1. **Graph structure:** first-order is a chain $X_{t-1}\to X_t$; second-order adds an edge so each
   node has two parents $X_{t-2}\to X_t \leftarrow X_{t-1}$.
2. **Conditional probability table:** first-order is indexed by one previous word (vocab-sized
   rows); second-order is indexed by **ordered pairs** of previous words (up to vocab² rows) — far
   larger.
3. **Context for prediction:** second-order uses the two preceding tokens, so it can disambiguate
   continuations that depend on a two-word history.
4. **Data needed:** second-order has many more parameters to estimate, so it needs substantially
   more data; from a tiny corpus most contexts are seen only once, making estimates sparse/peaked.


## Part XIII — Comparing the Two Models

In [10]:
def n_params(cpt):
    return sum(len(d) for d in cpt.values())

def n_zero_contexts(model_cpt, vocab):
    """Count (context) rows that are missing possible successors (illustrative sparsity)."""
    return sum(1 for d in model_cpt.values() if len(d) < len(vocab))

vocab = sorted({w for sent in corpus for w in sent})

# Diversity: unique sentences out of N samples.
rngA, rngB = random.Random(1), random.Random(1)
samp1 = {" ".join(lm1.generate(rngA)) for _ in range(50)}
samp2 = {" ".join(lm2.generate(rngB)) for _ in range(50)}

print(f"Vocabulary size                : {len(vocab)}")
print(f"First-order  distinct params   : {n_params(cpt1)}")
print(f"Second-order distinct params   : {n_params(lm2.cpt)}")
print(f"First-order  contexts (rows)   : {len(cpt1)}")
print(f"Second-order contexts (rows)   : {len(lm2.cpt)}")
print(f"Unique sentences / 50 (1st ord): {len(samp1)}")
print(f"Unique sentences / 50 (2nd ord): {len(samp2)}")
print("\nExample first-order sentences :", list(samp1)[:3])
print("Example second-order sentences:", list(samp2)[:3])


Vocabulary size                : 12
First-order  distinct params   : 17
Second-order distinct params   : 19
First-order  contexts (rows)   : 11
Second-order contexts (rows)   : 15
Unique sentences / 50 (1st ord): 21
Unique sentences / 50 (2nd ord): 6

Example first-order sentences : ['the cat sat on the cat sat on the mat', 'the cat sat on the dog ran to the dog sat on the dog sat on the mat', 'the cat ran to the cat sat on the mat']
Example second-order sentences: ['the cat ran to the park', 'the dog sat on the rug', 'the cat sat on the rug']


**Question 12 — Why does more context help, yet make estimation harder?**
More context (a longer history) lets the model capture dependencies that a shorter window misses,
so predictions can be more accurate and generated text more coherent. But the number of possible
contexts grows **exponentially** with the window length: a first-order CPT has $O(V)$ rows, a
second-order one $O(V^2)$, an $n$-th order one $O(V^{n})$. With limited data most of those contexts
are observed rarely or never, so the counts give noisy, over-confident (often zero) estimates.
This is the bias–variance / data-sparsity trade-off tied directly to CPT size.


## Part XIV — Connection to Modern Language Models

A modern autoregressive LM also models $P(X_t\mid X_1,\dots,X_{t-1})$, but instead of a hand-built
CPT it uses a **neural network** to estimate the conditional:
$x_1,\dots,x_{t-1} \to \text{neural network} \to P(X_t\mid x_1,\dots,x_{t-1})$, with
$P(x_1,\dots,x_T)=\prod_{t=1}^{T}P(x_t\mid x_{<t})$ and $P(x_1\mid\langle START\rangle)$ as the
first factor.

| | Simple BN model | Modern autoregressive LM |
|---|---|---|
| Representation | CPTs | Neural network |
| Context | Fixed window | Large learned context |
| Parameters | Explicit probabilities | Learned weights |
| Learning | Counting | Gradient-based training |
| Generation | Sampling | Sampling |

The **probabilistic objective is identical** — *predict the next token conditional on previous
tokens*; only the representation and learning method differ.


## Part XV & Final Reflection (Questions 13–14)

**Question 13 — Why is Approach B ("implement $P(X_t\mid X_{t-1})$ from transition counts with
sampling") preferable to Approach A ("write a language model for me")?**
Because it specifies the *intended behaviour and representation* before any code exists, which
lets you (a) know exactly what distribution should be estimated, (b) understand and inspect the
representation (counts → CPT), (c) **validate** the implementation against the specification,
(d) write **probabilistic invariant tests** (rows sum to 1), and (e) separate the *model* (the
mathematical object) from the *implementation* (the code). Approach A hands all of these
judgements to the LLM and gives you nothing concrete to verify against.

**Question 14 — What did thinking of the LM as a Bayesian network add?** (at least three)

* **A representation of dependencies** — explicit edges showing which tokens condition which.
* **A factorisation of the joint distribution** — the chain rule turns $P(\text{sentence})$ into
  a product of next-token conditionals.
* **A way to interpret conditional probabilities** — each CPT row *is* a next-word distribution.
* **A principled generation method** — ancestral sampling along the chain.
* **A way to reason about independence assumptions** — e.g. first- vs second-order Markov.
* **A way to understand the effect of increasing context** — larger parents ⇒ larger CPT ⇒ more
  data needed.
* **A way to test implementation against specification** — normalisation and parameter-count
  checks.


## Deliverables Summary

Produced in this notebook:

1. First-order model (`FirstOrderLM`) and 2. second-order model (`SecondOrderLM`).
3. Conditional probability tables for selected contexts (Parts IV, VIII, XI).
4. Examples of generated text (Parts IX, X, XI).
5. Probability-normalisation test results (Part VII, and for the 2nd-order model).
6. Answers to Questions 1–14 (inline).
7. Reflection on LLM use: we specified behaviour first (Approach B), implemented it ourselves, and
   validated it with the $\sum_v P(v\mid w)=1$ invariant — exactly the kind of property-based check
   that catches an LLM-generated normalisation bug (e.g. a row summing to 0.87).

> **Progression:** Probability → Bayesian Network → Autoregressive Model → Language Generation.
> The fundamental question throughout is $P(\text{next token}\mid\text{previous tokens})$.
